# 公开语料流程验证（可选）

先在项目根目录执行 `python tools/prepare_public_dataset.py`，阅读数据协议后使用。公开包不附原始语料。仅上传生成的 `evaluator/dataset/materials/` 中 5 个 TXT；参考答案留在评测端。填写自己的应用 URL，先跑 1 题 × 2 类行为 × 2 套应用。


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, os, sys, subprocess

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "evaluator" / "runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请在VS Code打开完整项目文件夹，再运行Notebook")
PACKAGE_DIR = PROJECT_ROOT / "evaluator"
if not (PACKAGE_DIR / "runner.py").is_file():
    raise FileNotFoundError(f"请解压完整evaluator文件夹到项目根目录：{PACKAGE_DIR}")

SUITE = "dataset"
DATASET_SUITE = PACKAGE_DIR / "dataset" / "tasks.json"
NEW_MATERIALS_READY = False       # 两边上传5个TXT、索引完成、绑定正确知识库并发布后改True
DATASET_SELECTION_REVIEWED = False  # 阅读下面5题预览和原协议，认可本次用途后改True
SELECTED_TASKS = ["DU001"]       # 首次1题；后续["DU002", "DU003", "DU004", "DU005"]；[]为全部
SELECTED_PRODUCTS = ["dify_app", "coze_app"]   # 只重测Coze可改为["coze_app"]
MAX_RUNS = 4                     # 其余4题双产品共16次时改16；全部5题改20
MAX_CALLS_PER_RUN = 26
OPERATION_MODE = "rules"   # rules：明确网页动作不调用模型；model：保留原控制模型模式
MAX_OPERATION_STEPS = 80 if OPERATION_MODE == "rules" else 12

products = {
    "dify_app": {"name": "Dify 自建知识库助手", "url": "https://example.com/REPLACE_WITH_DIFY_APP_URL",
                 "settings": {"model": "未记录", "retrieval": "未记录", "source_display": "未记录", "prompt_version": "未记录"}},
    "coze_app": {"name": "Coze 自建知识库助手", "url": "https://example.com/REPLACE_WITH_COZE_APP_URL",
                 "settings": {"model": "未记录", "retrieval": "未记录", "source_display": "未记录", "prompt_version": "未记录"}}
}
for product in products.values():
    product["material_confirmed"] = NEW_MATERIALS_READY
suite_path = DATASET_SUITE
suite = json.loads(suite_path.read_text(encoding="utf-8"))
manifest = []
for doc in suite["documents"]:
    path = (suite_path.parent / doc["path"]).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"缺少已准备的资料：{path}")
    manifest.append({"document_id": doc["id"], "path": str(path),
                     "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                     "remote_names": doc["remote_names"]})
experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_dir = PROJECT_ROOT / "evaluation" / "public_dataset_runs" / experiment_id
request_dir = PROJECT_ROOT / "evaluation" / "public_dataset_requests"
request_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = request_dir / f"{experiment_id}.json"
config = {"project_root": str(PROJECT_ROOT), "suite_path": str(suite_path),
          "material_manifest": manifest, "output_dir": str(output_dir), "products": products,
          "selected_products": SELECTED_PRODUCTS, "selected_tasks": SELECTED_TASKS,
          "max_runs": MAX_RUNS, "max_calls_per_run": MAX_CALLS_PER_RUN,
          "operation_mode": OPERATION_MODE, "max_operation_steps": MAX_OPERATION_STEPS,
          "dataset_selection_reviewed": DATASET_SELECTION_REVIEWED,
          "headless": False, "stop_on_error": True}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
CHILD_ENV = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
print("Python:", sys.executable)
print("需上传的文件夹：", PACKAGE_DIR / "dataset" / "materials")
print("配置请求：", CONFIG_PATH)
print("本次结果：", output_dir)

## 离线检查：运行程序与原始样本校验
不打开网页，不调用API。会校验官方包/开发集哈希、原文子串、材料与原文一致性，及运行逻辑。它不验证远端应用配置。

In [ ]:
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                       cwd=str(PACKAGE_DIR), env=CHILD_ENV, capture_output=True,
                       text=True, encoding="utf-8", errors="replace")
print(check.stdout)
print(check.stderr)
OFFLINE_CHECK_PASSED = check.returncode == 0
if not OFFLINE_CHECK_PASSED:
    raise RuntimeError("离线检查失败；请保留上方完整错误输出")
verified = subprocess.run([sys.executable, "verify_dataset.py"], cwd=str(PACKAGE_DIR), env=CHILD_ENV,
                          capture_output=True, text=True, encoding="utf-8", errors="replace")
print(verified.stdout, verified.stderr)
if verified.returncode:
    OFFLINE_CHECK_PASSED = False
    raise RuntimeError("原始样本校验失败")

## 本地样本预览：看资料支持和问题适配
下面包含参考答案，不要复制到产品提示词或上传知识库。答案是原文子串，本地计算的位置与官方原始`answer_start`分别记录。目的性5题只用于流程验证，不能代表整个数据集。

In [ ]:
reviews = json.loads((PACKAGE_DIR / "dataset/sample_review.json").read_text(encoding="utf-8"))
for item in reviews:
    print("\n", item["task_id"], item["original_question"])
    print("参考答案：", item["reference_answers"], "文档：", item["document_name"])
    for span in item["reference_spans"]:
        print("支持片段：", span["support_excerpt"])
        print("原文字符位置（从0开始，末尾不含）：", span["computed_context_start"], span["computed_context_end"],
              "官方原始位置：", span["original_answer_start"])
print("\n原协议：", PACKAGE_DIR / "dataset/raw/License.docx")
print("检查完成后在配置格确认两个Ready/Reviewed开关，再重新运行配置格。")

## 运行计划与消耗边界
默认4次；上限是测试Agent决策调用次数，不是tokens预算，也不包含被测产品内部模型消耗。

In [ ]:
preview = subprocess.run([sys.executable, "runner.py", "--plan", str(CONFIG_PATH)],
                         cwd=str(PACKAGE_DIR), env=CHILD_ENV, capture_output=True,
                         text=True, encoding="utf-8", errors="replace")
if preview.returncode:
    print(preview.stdout, preview.stderr)
    raise RuntimeError("计划不合法；检查选题、产品和MAX_RUNS")
planned = json.loads(preview.stdout)
print("计划网页体验次数：", planned["planned_runs"])
print("测试Agent决策调用总上限：", planned["max_decision_calls"])
for item in planned["jobs"]:
    print(item["task"], item["persona"], item["product"])

## 主动启动真实体验
确认两边上传的是`dataset/materials`、已发布且绑定正确资料；配置格的两个确认开关均为True，再把下面`RUN_LIVE`改为True。程序读取配置文件，请不要只改内存变量而遗漏重跑配置格。运行中不要关闭页面。

In [ ]:
RUN_LIVE = False
if not RUN_LIVE:
    print("未启动真实测试，没有调用产品或API。准备好后将RUN_LIVE改为True。")
else:
    if not globals().get("OFFLINE_CHECK_PASSED", False):
        raise RuntimeError("请先通过离线检查")
    saved_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    if saved_config.get("dataset_selection_reviewed") is not True:
        raise RuntimeError("阅读样本/原协议后确认DATASET_SELECTION_REVIEWED=True并重跑配置格")
    if any(saved_config["products"][p].get("material_confirmed") is not True for p in saved_config["selected_products"]):
        raise RuntimeError("上传、索引、绑定知识库并发布后设NEW_MATERIALS_READY=True并重跑配置格")
    if "LIVE_JOB" in globals() and LIVE_JOB.poll() is None:
        raise RuntimeError("已有任务正在运行，请查看下一格")
    if output_dir.exists():
        raise RuntimeError("此实验已运行过；重新运行配置格建立新实验，保留原结果")
    LOG_PATH = request_dir / f"{experiment_id}.log"
    with LOG_PATH.open("w", encoding="utf-8") as log:
        LIVE_JOB = subprocess.Popen([sys.executable, "runner.py", "--config", str(CONFIG_PATH)],
                                    cwd=str(PACKAGE_DIR), env=CHILD_ENV,
                                    stdout=log, stderr=subprocess.STDOUT)
    LOG_OFFSET = 0
    print("已启动，PID：", LIVE_JOB.pid)
    print("结果目录：", output_dir)

## 查看进度（可重复运行本格）
日志只显示新增内容；任务在后台运行，本格不等待整批完成。

In [ ]:
if "LIVE_JOB" not in globals():
    print("尚未启动真实体验。")
else:
    with LOG_PATH.open("r", encoding="utf-8", errors="replace") as log:
        log.seek(globals().get("LOG_OFFSET", 0))
        print(log.read())
        LOG_OFFSET = log.tell()
    status_path = output_dir / "batch_status.json"
    if status_path.is_file():
        try:
            print(json.loads(status_path.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            print("状态正在写入；稍后再运行本格。")
    print("进程返回码：", LIVE_JOB.poll())
    print("矩阵：", output_dir / "matrix.md")

## 人工复核与重建矩阵（不再调用产品）
编辑结果目录中的`human_reviews.json`，比较产品实际答案、`sample_review.json`和TXT；引用归属还需核对截图/页面全文。不要因文件名或词面命中直接填写passed。

`task_completion`和`source_authenticity`允许`not_reviewed`、`passed`、`failed`、`inconclusive`；通过/失败需填`reviewer`、`evidence_note`。`user_fit_note`说明哪种需求满足或仍有缺口。预期首答`needs_review`，表示规则未自动做语义裁判。

In [ ]:
REBUILD_REPORT = False
if not REBUILD_REPORT:
    print("请先编辑：", output_dir / "human_reviews.json")
else:
    if "LIVE_JOB" in globals() and LIVE_JOB.poll() is None:
        raise RuntimeError("请等真实体验结束后再重建报告")
    rebuilt = subprocess.run([sys.executable, "runner.py", "--rebuild", str(output_dir)],
                             cwd=str(PACKAGE_DIR), env=CHILD_ENV, capture_output=True,
                             text=True, encoding="utf-8", errors="replace")
    print(rebuilt.stdout, rebuilt.stderr)
    if rebuilt.returncode:
        raise RuntimeError("重建失败，请检查复核字段和证据要求")